In [ ]:
import os
print(os.getcwd())

In [ ]:
import sys
sys.path.append('./src')
import utils
import pandas as pd
import s3fs
import geopandas as gpd

arbres = pd.read_csv("https://www.data.gouv.fr/api/1/datasets/r/60433484-f30e-44ef-a362-e5553a9b7a42", sep = ";")

fs = s3fs.S3FileSystem(client_kwargs={"endpoint_url": "https://minio.lab.sspcloud.fr"})

MY_BUCKET = "raphcrre"
PATH_IRIS = f"{MY_BUCKET}/diffusion/projet_arbres/iris.gpkg"

with fs.open(PATH_IRIS, 'rb') as f:
    iris_france = gpd.read_file(f)

iris = iris_france[iris_france['code_insee'].astype(str).str.startswith('75')].copy()

df_arbres = utils.jointure_arbres_iris(arbres, iris)

df_arbres

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import utils3

# 1. Récupération des données via ton module
df_top = utils3.get_top_species(arbres, column_name = "LIBELLE FRANCAIS")

# 2. Création du graphique
plt.figure(figsize=(12, 6))
sns.set_theme(style="whitegrid")

ax = sns.barplot(
    data=df_top, 
    x='Nombre', 
    y='Espece', 
    palette='viridis'
)

# 3. Cosmétique (pour que ce soit "qualitatif")
plt.title('Top 10 des essences d\'arbres à Paris', fontsize=15, pad=20)
plt.xlabel('Nombre d\'arbres')
plt.ylabel('Essence (nom commun)')

# Ajout des pourcentages au bout des barres
for i, p in enumerate(ax.patches):
    percentage = f"{df_top['Pourcentage'].iloc[i]}%"
    ax.annotate(percentage, (p.get_width(), p.get_y() + p.get_height()/2),
                ha='left', va='center', xytext=(5, 0), textcoords='offset points')

plt.show()